# Instalación e imports

In [3]:
%pip install -q "modin[ray]" ray pyarrow gcsfs fsspec --root-user-action=ignore

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import subprocess
import numpy as np

os.environ["MODIN_ENGINE"] = "ray"
os.environ["MODIN_CPUS"] = "2"

import ray
import modin.pandas as pd

In [2]:
if ray.is_initialized():
    ray.shutdown()

ray.init(
    ignore_reinit_error=True,
    num_cpus=2,
    include_dashboard=False
)

2026-05-05 06:26:32,884	INFO worker.py:2012 -- Started a local Ray instance.


Python version:,3.11.14
Ray version:,2.55.1


### Rutas del proyecto

In [32]:
GCS_RAW_PATH = "gs://proyectobigdata2026/raw/restaurant_inspections.csv"

LOCAL_RAW_PATH = "/tmp/restaurant_inspections.csv"
LOCAL_PROCESSED_MODIN = "/tmp/modin_restaurant_inspections_clean"
GCS_PROCESSED_MODIN = "gs://proyectobigdata2026/processed/modin/restaurant_inspections_clean/"
GCS_OUTPUTS_MODIN = "gs://proyectobigdata2026/results/outputs/modin/"
GCS_KPIS_MODIN = "gs://proyectobigdata2026/results/kpis/modin/"

In [4]:
!gcloud storage ls gs://proyectobigdata2026/raw/

gs://proyectobigdata2026/raw/

gs://proyectobigdata2026/raw/:
gs://proyectobigdata2026/raw/
gs://proyectobigdata2026/raw/restaurant_inspections.csv


In [5]:
subprocess.run(
    f"gcloud storage cp {GCS_RAW_PATH} {LOCAL_RAW_PATH}",
    shell=True,
    check=True
)

Copying gs://proyectobigdata2026/raw/restaurant_inspections.csv to file:///tmp/restaurant_inspections.csv
  
.....

Average throughput: 224.6MiB/s


CompletedProcess(args='gcloud storage cp gs://proyectobigdata2026/raw/restaurant_inspections.csv /tmp/restaurant_inspections.csv', returncode=0)

In [6]:
dtype_config = {
    "camis": "object",
    "dba": "object",
    "boro": "object",
    "building": "object",
    "street": "object",
    "zipcode": "object",
    "phone": "object",
    "cuisine_description": "object",
    "inspection_date": "object",
    "action": "object",
    "violation_code": "object",
    "violation_description": "object",
    "critical_flag": "object",
    "score": "object",
    "grade": "object",
    "grade_date": "object",
    "record_date": "object",
    "inspection_type": "object",
    "latitude": "float64",
    "longitude": "float64",
    "community_board": "float64",
    "council_district": "float64",
    "census_tract": "float64",
    "bin": "float64",
    "bbl": "float64",
    "nta": "object",
    "location": "object"
}

df_raw = pd.read_csv(
    LOCAL_RAW_PATH,
    dtype=dtype_config
)

df_raw.head()

,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,...,inspection_type,longitude,latitude,council_district,bin,community_board,nta,census_tract,bbl,location
0,50180205,SOHO PIZZA,Queens,42-27,35 AVENUE,11101,3472422217,NaN,1900-01-01T00:00:00.000,NaN,...,NaN,-73.920247,40.754261,22.0,4307241.0,401.0,QN70,15900.0,4.006750e+09,POINT (-73.920246508724 40.754261466804)
1,41476556,POPEYES,Queens,165-25,LIBERTY AVENUE,11433,7185234233,Chicken,2025-02-28T00:00:00.000,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,-73.791514,40.702224,27.0,4216231.0,412.0,QN61,44400.0,4.101550e+09,POINT (-73.79151424692 40.702223564768)
2,50088657,MINNIE'S BAR,Brooklyn,885,4 AVENUE,11232,7188324855,American,2024-10-02T00:00:00.000,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,-74.002808,40.655845,38.0,3010178.0,307.0,BK32,8400.0,3.006850e+09,POINT (-74.002807564897 40.655845217215)
3,50047825,"L'AMICO, BACK BAR",Manhattan,839,6 AVENUE,10001,2122014065,American,2024-06-13T00:00:00.000,Violations were cited in the following area(s).,...,Cycle Inspection / Initial Inspection,-73.989906,40.746894,3.0,1015146.0,105.0,MN17,9500.0,1.008058e+09,POINT (-73.989905606376 40.746894352563)
4,50181761,DRAVIDA,Manhattan,211,1 AVENUE,10003,4016995421,Indian,2026-03-31T00:00:00.000,Violations were cited in the following area(s).,...,Pre-permit (Non-operational) / Initial Inspection,-73.983248,40.730420,2.0,1006491.0,103.0,MN22,4000.0,1.004540e+09,POINT (-73.983247537461 40.730419502848)


In [8]:
print("Filas originales:", len(df_raw))
print("Columnas:", len(df_raw.columns))

df_raw.dtypes

Filas originales: 296740
Columnas: 27


camis                     object
dba                       object
boro                      object
building                  object
street                    object
zipcode                   object
phone                     object
cuisine_description       object
inspection_date           object
action                    object
violation_code            object
violation_description     object
critical_flag             object
score                     object
grade                     object
grade_date                object
record_date               object
inspection_type           object
longitude                float64
latitude                 float64
council_district         float64
bin                      float64
community_board          float64
nta                       object
census_tract             float64
bbl                      float64
location                  object
dtype: object

## Normalización de columnas

In [9]:
def normalize_column_name(name):
    return (
        name.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

df_raw.columns = [normalize_column_name(c) for c in df_raw.columns]

df_raw.columns

Index(['camis', 'dba', 'boro', 'building', 'street', 'zipcode', 'phone',
       'cuisine_description', 'inspection_date', 'action', 'violation_code',
       'violation_description', 'critical_flag', 'score', 'grade',
       'grade_date', 'record_date', 'inspection_type', 'longitude', 'latitude',
       'council_district', 'bin', 'community_board', 'nta', 'census_tract',
       'bbl', 'location'],
      dtype='object')

### Perfilamiento inicial

In [10]:
df_raw[
    ["camis", "inspection_date", "score", "boro", "critical_flag", "violation_code"]
].isna().sum()

camis                  0
inspection_date        0
score              17061
boro                   0
critical_flag          0
violation_code      5833
dtype: int64

In [11]:
df_raw["boro"].value_counts(dropna=False)

the groupby keys will be sorted anyway, although the 'sort=False' was passed. See the following issue for more details: https://github.com/modin-project/modin/issues/3571.


boro
Manhattan        110189
Brooklyn          75544
Queens            73748
Bronx             27118
Staten Island      9811
0                   330
Name: count, dtype: int64

In [12]:
df_raw["grade"].value_counts(dropna=False)

grade
NaN    150586
A       99277
B       18577
C       13578
N        9545
Z        4300
P         877
Name: count, dtype: int64

## Limpieza base

In [13]:
df = df_raw.copy()

text_columns = [
    "camis", "dba", "boro", "building", "street", "zipcode",
    "phone", "cuisine_description", "action", "violation_code",
    "violation_description", "critical_flag", "grade",
    "inspection_type", "nta"
]

for c in text_columns:
    if c in df.columns:
        df[c] = df[c].astype("string").str.strip()

df["dba"] = df["dba"].str.upper()
df["grade"] = df["grade"].str.upper()

In [14]:
df["inspection_date"] = pd.to_datetime(
    df["inspection_date"],
    errors="coerce"
)

df["grade_date"] = pd.to_datetime(
    df["grade_date"],
    errors="coerce"
)

df["record_date"] = pd.to_datetime(
    df["record_date"],
    errors="coerce"
)

df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

In [15]:
df_clean = df[
    df["camis"].notna() &
    df["inspection_date"].notna() &
    (df["inspection_date"] != pd.Timestamp("1900-01-01"))
].copy()

print("Filas después de limpieza base:", len(df_clean))

Filas después de limpieza base: 293267


### Variables auxiliares

In [16]:
df_clean["year"] = df_clean["inspection_date"].dt.year
df_clean["month"] = df_clean["inspection_date"].dt.month

df_clean["season"] = "Otra temporada"
df_clean.loc[df_clean["month"].isin([6, 7, 8]), "season"] = "Verano"
df_clean.loc[df_clean["month"].isin([12, 1, 2]), "season"] = "Invierno"

In [17]:
df_clean["is_critical"] = (
    df_clean["critical_flag"]
    .str.lower()
    .eq("critical")
    .astype("int64")
)

df_clean["has_violation"] = df_clean["violation_code"].notna().astype("int64")

df_clean["inspection_id"] = (
    df_clean["camis"].astype(str) + "_" +
    df_clean["inspection_date"].dt.strftime("%Y-%m-%d") + "_" +
    df_clean["inspection_type"].fillna("unknown").astype(str)
)

In [18]:
df_clean["risk_level"] = "Sin puntaje"

df_clean.loc[
    df_clean["score"].notna() & (df_clean["score"] <= 13),
    "risk_level"
] = "Bajo"

df_clean.loc[
    df_clean["score"].notna() &
    (df_clean["score"] >= 14) &
    (df_clean["score"] <= 27),
    "risk_level"
] = "Medio"

df_clean.loc[
    df_clean["score"].notna() & (df_clean["score"] >= 28),
    "risk_level"
] = "Alto"

In [19]:
df_clean["sanitary_status"] = "Requiere revisión"

df_clean.loc[
    (df_clean["grade"] == "A") &
    (df_clean["is_critical"] == 0),
    "sanitary_status"
] = "Excelente"

df_clean.loc[
    (df_clean["grade"] == "A") &
    (df_clean["is_critical"] == 1),
    "sanitary_status"
] = "Aceptable con alerta"

df_clean.loc[
    df_clean["grade"].isin(["B", "C"]),
    "sanitary_status"
] = "Riesgo moderado"

### Corrección de boro usando zipcode

In [20]:
valid_boro_map = (
    df_clean[
        df_clean["zipcode"].notna() &
        df_clean["boro"].notna() &
        (~df_clean["boro"].isin(["0", "Missing", "MISSING", "missing"]))
    ][["zipcode", "boro"]]
    .drop_duplicates(subset=["zipcode"])
)

valid_boro_map = valid_boro_map.rename(columns={"boro": "boro_from_zip"})

df_clean = df_clean.merge(
    valid_boro_map,
    on="zipcode",
    how="left"
)

In [21]:
invalid_boro = (
    df_clean["boro"].isna() |
    df_clean["boro"].isin(["0", "Missing", "MISSING", "missing"])
)

df_clean["boro_corrected"] = df_clean["boro"]

df_clean.loc[invalid_boro, "boro_corrected"] = df_clean.loc[
    invalid_boro,
    "boro_from_zip"
]

df_clean["boro_corrected"] = df_clean["boro_corrected"].fillna("Sin datos")

df_clean = df_clean.drop(columns=["boro_from_zip"])

### Imputación de grade según score

In [26]:
def grade_from_score(score):
    if pd.isna(score):
        return None
    if score <= 13:
        return "A"
    if score <= 27:
        return "B"
    return "C"

df_clean["grade_imputed"] = df_clean["grade"]

missing_grade = df_clean["grade_imputed"].isna()

df_clean.loc[missing_grade, "grade_imputed"] = df_clean.loc[
    missing_grade,
    "score"
].apply(grade_from_score)

print("Nulos en grade original:", df_clean["grade"].isna().sum())
print("Nulos en grade_imputed:", df_clean["grade_imputed"].isna().sum())

Nulos en grade original: 147113
Nulos en grade_imputed: 13577


In [27]:
df_clean["has_inspection"] = df_clean["inspection_date"].notna()

df_clean["closure_flag"] = (
    df_clean["action"]
    .fillna("")
    .str.lower()
    .str.contains("closed", na=False)
)

critical_norm = df_clean["critical_flag"].fillna("").str.lower()

df_clean["severity_score"] = np.select(
    [
        critical_norm.eq("critical"),
        critical_norm.eq("not critical")
    ],
    [
        2,
        1
    ],
    default=0
)

# Base final para análisis

In [28]:
df_analysis = df_clean[
    df_clean["boro_corrected"].notna() &
    (df_clean["boro_corrected"] != "Sin datos") &
    df_clean["score"].notna() &
    df_clean["inspection_date"].notna() &
    df_clean["action"].notna()
].copy()

print("Filas originales:", len(df_raw))
print("Filas después de limpieza base:", len(df_clean))
print("Filas válidas para análisis:", len(df_analysis))

Filas originales: 296740
Filas después de limpieza base: 293267
Filas válidas para análisis: 279493


In [29]:
df_analysis[
    [
        "camis", "dba", "boro_corrected", "cuisine_description",
        "inspection_date", "score", "grade", "grade_imputed",
        "critical_flag", "risk_level", "sanitary_status"
    ]
].head()

,camis,dba,boro_corrected,cuisine_description,inspection_date,score,grade,grade_imputed,critical_flag,risk_level,sanitary_status
0,41476556,POPEYES,Queens,Chicken,2025-02-28,17.0,<NA>,B,Critical,Medio,Requiere revisión
1,50088657,MINNIE'S BAR,Brooklyn,American,2024-10-02,26.0,<NA>,B,Critical,Medio,Requiere revisión
2,50047825,"L'AMICO, BACK BAR",Manhattan,American,2024-06-13,37.0,<NA>,C,Critical,Alto,Requiere revisión
3,50181761,DRAVIDA,Manhattan,Indian,2026-03-31,42.0,<NA>,C,Not Critical,Alto,Requiere revisión
4,50132762,MIKADO,Queens,Japanese,2023-07-27,25.0,<NA>,B,Critical,Medio,Requiere revisión


### Checkpoint Modin en Parquet

In [33]:
import shutil
import os
import subprocess

# Limpiar salida local anterior
if os.path.exists(LOCAL_PROCESSED_MODIN):
    shutil.rmtree(LOCAL_PROCESSED_MODIN)

df_analysis.to_parquet(
    LOCAL_PROCESSED_MODIN,
    engine="pyarrow",
    index=False
)

In [34]:
subprocess.run(
    f"gcloud storage rm -r {GCS_PROCESSED_MODIN} || true",
    shell=True
)

subprocess.run(
    f"gcloud storage cp -r {LOCAL_PROCESSED_MODIN}/* {GCS_PROCESSED_MODIN}",
    shell=True,
    check=True
)

Removing objects:
  

Removing managed folders:
  
failed.
ERROR: (gcloud.storage.rm) The following URLs matched no objects or files:
gs://proyectobigdata2026/processed/modin/restaurant_inspections_clean/
Copying file:///tmp/modin_restaurant_inspections_clean/part-0000.snappy.parquet to gs://proyectobigdata2026/processed/modin/restaurant_inspections_clean/part-0000.snappy.parquet
Copying file:///tmp/modin_restaurant_inspections_clean/part-0001.snappy.parquet to gs://proyectobigdata2026/processed/modin/restaurant_inspections_clean/part-0001.snappy.parquet
  
...

Average throughput: 356.7MiB/s


CompletedProcess(args='gcloud storage cp -r /tmp/modin_restaurant_inspections_clean/* gs://proyectobigdata2026/processed/modin/restaurant_inspections_clean/', returncode=0)

In [35]:
!gcloud storage ls gs://proyectobigdata2026/processed/modin/restaurant_inspections_clean/

gs://proyectobigdata2026/processed/modin/restaurant_inspections_clean/part-0000.snappy.parquet
gs://proyectobigdata2026/processed/modin/restaurant_inspections_clean/part-0001.snappy.parquet


In [36]:
df_analysis = pd.read_parquet(LOCAL_PROCESSED_MODIN)

df_analysis.head()

,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,...,is_critical,has_violation,inspection_id,risk_level,sanitary_status,boro_corrected,has_inspection,closure_flag,severity_score,grade_imputed
0,41476556,POPEYES,Queens,165-25,LIBERTY AVENUE,11433,7185234233,Chicken,2025-02-28,Violations were cited in the following area(s).,...,1,1,41476556_2025-02-28_Cycle Inspection / Initial...,Medio,Requiere revisión,Queens,True,False,2,B
1,50088657,MINNIE'S BAR,Brooklyn,885,4 AVENUE,11232,7188324855,American,2024-10-02,Violations were cited in the following area(s).,...,1,1,50088657_2024-10-02_Cycle Inspection / Initial...,Medio,Requiere revisión,Brooklyn,True,False,2,B
2,50047825,"L'AMICO, BACK BAR",Manhattan,839,6 AVENUE,10001,2122014065,American,2024-06-13,Violations were cited in the following area(s).,...,1,1,50047825_2024-06-13_Cycle Inspection / Initial...,Alto,Requiere revisión,Manhattan,True,False,2,C
3,50181761,DRAVIDA,Manhattan,211,1 AVENUE,10003,4016995421,Indian,2026-03-31,Violations were cited in the following area(s).,...,0,1,50181761_2026-03-31_Pre-permit (Non-operationa...,Alto,Requiere revisión,Manhattan,True,False,1,C
4,50132762,MIKADO,Queens,116-09,METROPOLITAN AVENUE,11418,7185590088,Japanese,2023-07-27,Violations were cited in the following area(s).,...,1,1,50132762_2023-07-27_Pre-permit (Operational) /...,Medio,Requiere revisión,Queens,True,False,2,B


# CRUD con Modin

## MODIN-C1: Create — variable has_inspection

In [50]:
crud_c1_has_inspection = (
    df_analysis["has_inspection"]
    .value_counts(dropna=False)
    .reset_index()
)

crud_c1_has_inspection.columns = ["has_inspection", "total_records"]

crud_c1_has_inspection

,has_inspection,total_records
0,True,279493


## MODIN-R1: Read — restaurantes de alto riesgo

In [37]:
crud_r1_high_risk = df_analysis[
    df_analysis["score"] > 30
][
    [
        "camis", "dba", "boro_corrected",
        "zipcode", "cuisine_description",
        "inspection_date", "score", "grade_imputed"
    ]
].sort_values("score", ascending=False)

crud_r1_high_risk.head(20)

,camis,dba,boro_corrected,zipcode,cuisine_description,inspection_date,score,grade_imputed
148775,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
127304,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
75560,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
91715,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
254168,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
92849,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
199298,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
119590,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
213684,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N
212599,50182863,TAPAS ON LEX,Manhattan,10029,Latin American,2026-04-01,214.0,N


## MODIN-R2: Read — restaurantes por tipo de cocina

In [38]:
crud_r2_cuisine_summary = (
    df_analysis
    .groupby("cuisine_description")["camis"]
    .nunique()
    .reset_index()
)

crud_r2_cuisine_summary.columns = [
    "cuisine_description",
    "unique_restaurants"
]

crud_r2_cuisine_summary = crud_r2_cuisine_summary.sort_values(
    "unique_restaurants",
    ascending=False
)

crud_r2_cuisine_summary.head(20)

,cuisine_description,unique_restaurants
2,American,4914
19,Chinese,2279
22,Coffee/Tea,2217
67,Pizza,1614
56,Mexican,1093
53,Latin American,1067
49,Japanese,1060
48,Italian,1008
7,Bakery Products/Desserts,924
15,Caribbean,758


## MODIN-U1: Update — imputación de grade

In [39]:
crud_u1_grade_imputation = pd.DataFrame({
    "metric": [
        "missing_grade_original",
        "missing_grade_after_imputation",
        "records_imputed"
    ],
    "value": [
        int(df_analysis["grade"].isna().sum()),
        int(df_analysis["grade_imputed"].isna().sum()),
        int((df_analysis["grade"].isna() & df_analysis["grade_imputed"].notna()).sum())
    ]
})

crud_u1_grade_imputation

,metric,value
0,missing_grade_original,133462
1,missing_grade_after_imputation,0
2,records_imputed,133462


## MODIN-D1: Delete lógico — registros válidos para análisis

In [40]:
crud_d1_delete_summary = pd.DataFrame({
    "stage": [
        "dataset_original",
        "dataset_clean",
        "dataset_analysis"
    ],
    "total_rows": [
        len(df_raw),
        len(df_clean),
        len(df_analysis)
    ]
})

crud_d1_delete_summary["rows_removed_vs_original"] = (
    crud_d1_delete_summary["total_rows"].iloc[0] -
    crud_d1_delete_summary["total_rows"]
)

crud_d1_delete_summary

Please refer to https://modin.readthedocs.io/en/stable/supported_apis/defaulting_to_pandas.html for explanation.


,stage,total_rows,rows_removed_vs_original
0,dataset_original,296740,0
1,dataset_clean,293267,3473
2,dataset_analysis,279493,17247


# KPIs con Modin

## KPI 8: Índice de severidad de violación

In [41]:
kpi8_global = pd.DataFrame({
    "metric": ["global_severity_index"],
    "value": [round(df_analysis["severity_score"].mean(), 4)]
})

kpi8_global

,metric,value
0,global_severity_index,1.5524


In [42]:
kpi8_by_cuisine = (
    df_analysis
    .groupby("cuisine_description")["severity_score"]
    .mean()
    .reset_index()
)

kpi8_by_cuisine.columns = [
    "cuisine_description",
    "avg_severity_score"
]

kpi8_by_cuisine = kpi8_by_cuisine.sort_values(
    "avg_severity_score",
    ascending=False
)

kpi8_by_cuisine.head(20)

,cuisine_description,avg_severity_score
26,Czech,1.687500
8,Bangladeshi,1.621455
24,Creole,1.609639
21,Chinese/Japanese,1.598485
64,Pakistani,1.598230
23,Continental,1.596737
1,African,1.593653
44,Indian,1.589995
68,Polish,1.589595
3,Armenian,1.586667


In [43]:
kpi8_by_boro = (
    df_analysis
    .groupby("boro_corrected")["severity_score"]
    .mean()
    .reset_index()
)

kpi8_by_boro.columns = [
    "boro_corrected",
    "avg_severity_score"
]

kpi8_by_boro = kpi8_by_boro.sort_values(
    "avg_severity_score",
    ascending=False
)

kpi8_by_boro

,boro_corrected,avg_severity_score
4,Staten Island,1.572555
3,Queens,1.561649
1,Brooklyn,1.552687
0,Bronx,1.547924
2,Manhattan,1.545339


## KPI 9: Reincidencia geográfica

In [44]:
df_critical = df_analysis[df_analysis["is_critical"] == 1]

critical_counts = (
    df_critical
    .groupby("camis")["inspection_date"]
    .nunique()
    .reset_index()
)

critical_counts.columns = [
    "camis",
    "critical_inspection_count"
]

critical_counts["is_recurrent"] = (
    critical_counts["critical_inspection_count"] > 1
)

In [45]:
restaurant_zip_catalog = (
    df_analysis
    .dropna(subset=["zipcode"])
    .groupby("camis")["zipcode"]
    .first()
    .reset_index()
)

df_recurrence_zip = restaurant_zip_catalog.merge(
    critical_counts[["camis", "is_recurrent"]],
    on="camis",
    how="left"
)

df_recurrence_zip["is_recurrent"] = (
    df_recurrence_zip["is_recurrent"]
    .fillna(False)
)

In [46]:
kpi9_recurrence_zip = (
    df_recurrence_zip
    .groupby("zipcode")
    .agg({
        "camis": "count",
        "is_recurrent": "sum"
    })
    .reset_index()
)

kpi9_recurrence_zip.columns = [
    "zipcode",
    "total_restaurants",
    "recurrent_restaurants"
]

kpi9_recurrence_zip = kpi9_recurrence_zip[
    kpi9_recurrence_zip["total_restaurants"] >= 10
]

kpi9_recurrence_zip["recurrence_rate_percentage"] = (
    kpi9_recurrence_zip["recurrent_restaurants"] /
    kpi9_recurrence_zip["total_restaurants"] *
    100
).round(2)

kpi9_recurrence_zip = kpi9_recurrence_zip.sort_values(
    "recurrence_rate_percentage",
    ascending=False
)

kpi9_recurrence_zip.head(10)

,zipcode,total_restaurants,recurrent_restaurants,recurrence_rate_percentage
192,11411,22,21,95.45
37,10039,19,18,94.74
153,11228,62,58,93.55
111,10470,41,37,90.24
54,10118,10,9,90.00
107,10466,85,76,89.41
169,11356,62,55,88.71
89,10310,61,54,88.52
94,10453,94,83,88.30
115,10474,32,28,87.50


## KPI 10: Índice de presión de inspección

In [47]:
kpi10_inspection_pressure = (
    df_analysis
    .groupby("boro_corrected")
    .agg({
        "inspection_id": "nunique",
        "camis": "nunique"
    })
    .reset_index()
)

kpi10_inspection_pressure.columns = [
    "boro_corrected",
    "total_inspections",
    "unique_restaurants"
]

kpi10_inspection_pressure["inspection_pressure"] = (
    kpi10_inspection_pressure["total_inspections"] /
    kpi10_inspection_pressure["unique_restaurants"]
).round(2)

kpi10_inspection_pressure = kpi10_inspection_pressure.sort_values(
    "inspection_pressure",
    ascending=False
)

kpi10_inspection_pressure

,boro_corrected,total_inspections,unique_restaurants,inspection_pressure
0,Bronx,7691,2412,3.19
1,Brooklyn,21760,7029,3.10
3,Queens,19508,6288,3.10
4,Staten Island,3001,1006,2.98
2,Manhattan,31695,10834,2.93


# Guardado de resultados en GCS

In [48]:
def save_modin_csv_to_gcs(df_result, gcs_path, index=False):
    local_tmp = "/tmp/" + os.path.basename(gcs_path)

    df_result.to_csv(local_tmp, index=index)

    subprocess.run(
        f"gcloud storage cp {local_tmp} {gcs_path}",
        shell=True,
        check=True
    )

    os.remove(local_tmp)
    print(f"Guardado en: {gcs_path}")

## Guardar CRUD

In [51]:
save_modin_csv_to_gcs(
    crud_c1_has_inspection,
    GCS_OUTPUTS_MODIN + "modin_crud_c1_has_inspection.csv"
)

save_modin_csv_to_gcs(
    crud_r1_high_risk.head(1000),
    GCS_OUTPUTS_MODIN + "modin_crud_r1_high_risk_restaurants.csv"
)

save_modin_csv_to_gcs(
    crud_r2_cuisine_summary,
    GCS_OUTPUTS_MODIN + "modin_crud_r2_cuisine_summary.csv"
)

save_modin_csv_to_gcs(
    crud_u1_grade_imputation,
    GCS_OUTPUTS_MODIN + "modin_crud_u1_grade_imputation.csv"
)

save_modin_csv_to_gcs(
    crud_d1_delete_summary,
    GCS_OUTPUTS_MODIN + "modin_crud_d1_delete_summary.csv"
)

Copying file:///tmp/modin_crud_c1_has_inspection.csv to gs://proyectobigdata2026/results/outputs/modin/modin_crud_c1_has_inspection.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/modin/modin_crud_c1_has_inspection.csv


Copying file:///tmp/modin_crud_r1_high_risk_restaurants.csv to gs://proyectobigdata2026/results/outputs/modin/modin_crud_r1_high_risk_restaurants.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/modin/modin_crud_r1_high_risk_restaurants.csv


Copying file:///tmp/modin_crud_r2_cuisine_summary.csv to gs://proyectobigdata2026/results/outputs/modin/modin_crud_r2_cuisine_summary.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/modin/modin_crud_r2_cuisine_summary.csv


Copying file:///tmp/modin_crud_u1_grade_imputation.csv to gs://proyectobigdata2026/results/outputs/modin/modin_crud_u1_grade_imputation.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/modin/modin_crud_u1_grade_imputation.csv


Copying file:///tmp/modin_crud_d1_delete_summary.csv to gs://proyectobigdata2026/results/outputs/modin/modin_crud_d1_delete_summary.csv
  
.


Guardado en: gs://proyectobigdata2026/results/outputs/modin/modin_crud_d1_delete_summary.csv


## Guardar KPIs

In [52]:
save_modin_csv_to_gcs(
    kpi8_global,
    GCS_KPIS_MODIN + "modin_kpi8_global_severity.csv"
)

save_modin_csv_to_gcs(
    kpi8_by_cuisine,
    GCS_KPIS_MODIN + "modin_kpi8_severity_by_cuisine.csv"
)

save_modin_csv_to_gcs(
    kpi8_by_boro,
    GCS_KPIS_MODIN + "modin_kpi8_severity_by_boro.csv"
)

save_modin_csv_to_gcs(
    kpi9_recurrence_zip,
    GCS_KPIS_MODIN + "modin_kpi9_geographic_recurrence.csv"
)

save_modin_csv_to_gcs(
    kpi10_inspection_pressure,
    GCS_KPIS_MODIN + "modin_kpi10_inspection_pressure.csv"
)

Copying file:///tmp/modin_kpi8_global_severity.csv to gs://proyectobigdata2026/results/kpis/modin/modin_kpi8_global_severity.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/modin/modin_kpi8_global_severity.csv


Copying file:///tmp/modin_kpi8_severity_by_cuisine.csv to gs://proyectobigdata2026/results/kpis/modin/modin_kpi8_severity_by_cuisine.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/modin/modin_kpi8_severity_by_cuisine.csv


Copying file:///tmp/modin_kpi8_severity_by_boro.csv to gs://proyectobigdata2026/results/kpis/modin/modin_kpi8_severity_by_boro.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/modin/modin_kpi8_severity_by_boro.csv


Copying file:///tmp/modin_kpi9_geographic_recurrence.csv to gs://proyectobigdata2026/results/kpis/modin/modin_kpi9_geographic_recurrence.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/modin/modin_kpi9_geographic_recurrence.csv


Copying file:///tmp/modin_kpi10_inspection_pressure.csv to gs://proyectobigdata2026/results/kpis/modin/modin_kpi10_inspection_pressure.csv
  
.


Guardado en: gs://proyectobigdata2026/results/kpis/modin/modin_kpi10_inspection_pressure.csv
